# Cast

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389.
Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
import torch
from IPython.display import SVG, display

from examples.algorithms.binary_field_arithmetic import FieldOperation
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum import Circuit
from flagquantum.core.ir.program.register_types import fixed_point_register_type, signed_register_type
from flagquantum.core.ir.program.registers import Register, RegisterSide
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.simulation.fixed_point_encoding import fixed_point_real_value

## Reinterpret a register

Cast preserves the stored bits and changes their interpretation. The
input and output have the same word width. Array elements keep
row-major order and bits within a word keep big-endian order. Quantum
and classical storage are distinct. The optional metadata opt-in for
mixed storage does not model measurement or its decoherence.

The source example views a signed 32-bit word as an unsigned fixed-point
fraction with 32 fractional bits. Show the original typed signature and
bind its physical wire order directly. Typed Program execution has its
own dtype admission rules.

In [ ]:
from flagquantum.compiler.registers.layout.register_cast import RegisterCast

### Example Instances

In [ ]:
signed = signed_register_type(32)
fixed = fixed_point_register_type(32, 32)
cast = RegisterCast(32, 32)
signature = Signature((
    Register("reg", signed, side=RegisterSide.LEFT),
    Register("reg", fixed, side=RegisterSide.RIGHT),
))
wires = tuple(reversed(range(32)))
assert cast.bind(wires) == wires
stored = cast.reinterpret_integer(-1, source_signed=True)
assert fixed_point_real_value(stored, 32, 32) == 1 - 2**-32

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("Cast signed to fixed point", signature, {})])))

### Check the zero-gate view

Use a small coherent register to check the corresponding physical
operation. The view preserves each amplitude and uses no gates,
measurements, or temporary wires.

In [ ]:
small = RegisterCast(3, 3)
assert small.bind((2, 0, 1)) == (2, 0, 1)
values = torch.arange(1, 9, dtype=torch.float64)
initial = torch.complex(values, values.flip(0))
initial /= initial.norm()
circuit = Circuit(3, inputs=initial, dtype=initial.dtype)
circuit.append(FieldOperation(3, ()))
assert circuit.to_ir().instructions == ()
torch.testing.assert_close(circuit.state().flatten(), initial, rtol=0, atol=2e-12)